
# ColPali — OCR-Free Document Retrieval

**Day 3 — RAG & Agents · Practical 4 of 6 · Companion to the "Multi-modal & Vectorless RAG"
deck**

> **Running in Google Colab:** requires a **GPU runtime** (Runtime -> Change runtime type ->
> T4 GPU). Per feasibility research, ColPali fits comfortably on a free T4 (~2-4GB VRAM with
> quantization) — this notebook deliberately uses a **small page count (3-5 pages)** rather
> than a full corpus, since ColQwen2-class models can run out of memory on free T4 at scale.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Retrieve directly from PAGE IMAGES using ColPali — no OCR step, no text extraction
2. See late-interaction (MaxSim) scoring work on real document pages
3. Compare this against what a traditional OCR-based pipeline would have to do first

## Why This Matters for a Law Firm

Real contracts, exhibits, and scanned filings are rarely clean text. This notebook demonstrates
retrieving directly from page IMAGES — tables, signature blocks, and layout included — with no
OCR error-cascade risk, exactly the deck's core argument.

## Notebook Workflow

```mermaid
flowchart LR
    A["Contract pages\n(as images)"] --> B["ColPali\n(vision-language model)"]
    B --> C["Per-page patch\nembeddings"]
    D["Text query"] --> E["Query token\nembeddings"]
    C --> F["MaxSim late\ninteraction scoring"]
    E --> F
    F --> G["Most relevant\npage(s)"]



## Section 1 — Setup

`colpali-engine` provides the ColPali model and scoring utilities. This will download model
weights on first run (a few GB) — expect the first cell to take a few minutes.


In [ ]:

%pip install -q colpali-engine pdf2image pillow

import torch
from colpali_engine.models import ColPali, ColPaliProcessor

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")



## Section 2 — Load ColPali

Loaded in bfloat16 to keep VRAM usage low, matching the feasibility research's ~2-4GB estimate.


In [ ]:

MODEL_NAME = "vidore/colpali-v1.3"

colpali_model = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

colpali_processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

print("ColPali loaded.")



## Section 3 — Sample "Contract Pages"

In a real workflow, these would be scanned pages from an actual PDF (via `pdf2image`). To keep
this notebook fully self-contained and not dependent on an external file upload, we generate a
small set of synthetic "page images" -- rendered text laid out to resemble a contract page,
including a simple table, which is exactly the kind of content OCR struggles with and ColPali
handles natively.

**To use your own document:** replace this cell with
`from pdf2image import convert_from_path; pages = convert_from_path("your_contract.pdf")`,
capped to 3-5 pages per the GPU note above.


In [ ]:

from PIL import Image, ImageDraw, ImageFont

def render_page(title, body_lines, table_rows=None, size=(612, 792)):
    # Render a simple synthetic "contract page" image -- stands in for a real scanned page.
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)
    y = 40
    draw.text((40, y), title, fill="black")
    y += 40
    for line in body_lines:
        draw.text((40, y), line, fill="black")
        y += 25
    if table_rows:
        y += 20
        for row in table_rows:
            draw.text((40, y), row, fill="black")
            y += 22
    return img

pages = [
    render_page(
        "SECTION 7 -- INDEMNIFICATION",
        [
            "The Contractor shall indemnify, defend, and hold harmless the",
            "Client from any claims arising from gross negligence or",
            "willful misconduct in the performance of this Agreement.",
        ],
    ),
    render_page(
        "SCHEDULE A -- FEE TABLE",
        ["The following fees apply under this Agreement:"],
        table_rows=[
            "Service Tier      Monthly Fee     Setup Fee",
            "Standard           $2,500          $500",
            "Premium            $5,000          $1,000",
            "Enterprise         $12,000         $2,500",
        ],
    ),
    render_page(
        "SECTION 16 -- GOVERNING LAW",
        [
            "This Agreement shall be governed by and construed in",
            "accordance with the laws of the State of Delaware,",
            "without regard to conflict of laws principles.",
        ],
    ),
]

print(f"Rendered {len(pages)} sample pages.")
pages[1]  # display the fee table page inline



## Section 4 — Embed Pages (No OCR)

Each page image goes directly into ColPali -- no text extraction happens anywhere in this cell.


In [ ]:

def embed_pages(images):
    batch = colpali_processor.process_images(images).to(colpali_model.device)
    with torch.no_grad():
        embeddings = colpali_model(**batch)
    return embeddings

page_embeddings = embed_pages(pages)
print(f"Embedded {len(pages)} pages.")
print(f"Embedding shape per page (patches x dim): {page_embeddings.shape}")



## Section 5 — Embed a Text Query

The query stays plain text -- ColPali's asymmetry is exactly this: pages as images, queries as
text, scored against each other via late interaction.


In [ ]:

def embed_query(query_text):
    batch = colpali_processor.process_queries([query_text]).to(colpali_model.device)
    with torch.no_grad():
        embeddings = colpali_model(**batch)
    return embeddings

query = "What is the monthly fee for the Premium tier?"
query_embedding = embed_query(query)
print(f"Query embedding shape: {query_embedding.shape}")



## Section 6 — Score with Late Interaction (MaxSim)

Exactly the deck's mechanism: each query token embedding finds its best-matching page patch
embedding, and those best-matches sum into a per-page relevance score.


In [ ]:

scores = colpali_processor.score_multi_vector(query_embedding, page_embeddings)

page_titles = ["Page 1: Indemnification", "Page 2: Fee Table", "Page 3: Governing Law"]
ranked = sorted(zip(page_titles, scores[0].tolist()), key=lambda x: -x[1])

print(f"Query: {query!r}\n")
print("Page relevance scores (highest first):\n")
for title, score in ranked:
    print(f"  {title:<30} score={score:.4f}")



**Reading this result:** the Fee Table page should score highest -- notice it's a TABLE, not
prose. A traditional OCR pipeline would need to correctly detect the table region, OCR each
cell, and reconstruct row/column relationships before this query could even be attempted.
ColPali skips all of that entirely.



## Section 7 — Try It Yourself

Try a query targeting the other pages, and one that's deliberately ambiguous across pages.


In [ ]:

for test_query in [
    "What state's law governs this contract?",
    "Who is responsible for indemnification?",
]:
    q_emb = embed_query(test_query)
    test_scores = colpali_processor.score_multi_vector(q_emb, page_embeddings)
    best_page = page_titles[test_scores[0].argmax().item()]
    print(f"Query: {test_query!r}  ->  Best match: {best_page}")



## Key Takeaways

1. **No OCR step ran anywhere in this notebook** -- pages were embedded as images directly, yet
   retrieval correctly distinguished a table page from prose pages based on the query's intent.
2. **Late interaction (MaxSim)** is the same core mechanism as ColBERT (Retrieval Techniques
   notebook), just extended from text tokens to image patches.
3. **Real usage caps page count on free-tier GPUs** -- this notebook used 3 tiny synthetic
   pages specifically to stay within free Colab T4 memory; a real corpus of dozens/hundreds of
   pages would need a paid GPU tier or a more memory-efficient serving setup.

**Next up:** the *Minimal LangGraph ReAct Agent* notebook — moving from retrieval into agents.
